In [ ]:

import torch
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from torch import Tensor
from torch.nn import Sequential, Linear, ReLU, CrossEntropyLoss, Softplus, PReLU
from torch.utils.data import DataLoader
from torch.optim import Adam
import pandas as  pd

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path

# funciona com o notebook aberto em ml/ ou na raiz do projeto
CSV = next(p for p in (Path("dados/transacoes_rotuladas.csv"), Path("ml/dados/transacoes_rotuladas.csv")) if p.exists())

# keep_default_na=False: contraparte vazia continua "" (sem contraparte), não vira NaN
df = pd.read_csv(CSV, keep_default_na=False, parse_dates=["data"])
df["valor"] = df["valor"].astype(float)
print(df.shape)
df.dtypes

In [ ]:
ALVO = "categoria"

y = df[ALVO]
X = df.drop(columns=[ALVO])

print("X:", X.shape, "| y:", y.shape)
y.value_counts()

In [ ]:
# --- variáveis categóricas e numéricas -> matriz só de números para os modelos ---
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, LabelEncoder
# id_usuario não entra no modelo (usuários futuros não estão no treino):
# fica guardado para a validação cruzada por usuário (GroupKFold)
grupos = X["id_usuario"]

CATEGORICAS = ["historico", "meio", "tipo", "banco", "contraparte"]
NUMERICAS = ["valor_log", "dia_semana", "dia_mes", "mes"]

feat = X[CATEGORICAS].copy()
feat["valor_log"] = np.log1p(X["valor"].abs())  # o sinal já está em "tipo"; o log reduz a cauda longa
feat["dia_semana"] = X["data"].dt.day_of_week     # 0 = segunda ... 6 = domingo
feat["dia_mes"] = X["data"].dt.day
feat["mes"] = X["data"].dt.month

preprocessador = ColumnTransformer(
    [
        # one-hot: uma coluna 0/1 por valor; o que aparece menos de 5 vezes vira uma coluna "infrequent"
        # (importante para contraparte, com ~1.200 valores, e para lojas novas na aplicação)
        ("cat", OneHotEncoder(min_frequency=2, handle_unknown="infrequent_if_exist", sparse_output=False),
         CATEGORICAS),
        # min-max: numéricas na faixa [0, 1]
        ("num", MinMaxScaler(), NUMERICAS),
    ],
    verbose_feature_names_out=False,
).set_output(transform="pandas")

X = preprocessador.fit_transform(feat)

print("X transformado:", X.shape)
for nome, _, cols in preprocessador.transformers_:
    if nome == "cat":
        enc = preprocessador.named_transformers_["cat"]
        for c, cats in zip(cols, enc.categories_):
            print(f"  {c}: {len(cats)} valores -> {sum(1 for f in X.columns if f.startswith(c + '_'))} colunas")
    elif nome == "num":
        print(f"  numéricas: {len(cols)} colunas")

In [ ]:
# === PREPARAÇÃO DOS DADOS PARA A REDE NEURAL (não monta a rede) ===
# usa o `df` lido na célula do CSV
from itertools import combinations

import numpy as np
import pandas as pd
import torch
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, OneHotEncoder
from torch.utils.data import DataLoader, TensorDataset

# ---- configuração ----
DIVISAO = "usuario"   # "usuario": teste só com usuários que a rede nunca viu (caso real) | "aleatorio"
USAR_TFIDF = True     # True: contraparte em pedaços de 3-5 letras (TF-IDF) | False: contraparte no one-hot
USAR_DATA = False     # dia_semana / dia_mes / mes (nos testes, a data não ajudou)
BATCH = 256
SEMENTE = 42

torch.manual_seed(SEMENTE)
np.random.seed(SEMENTE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- atributos (colunas ainda em texto/números brutos) ----
base = df[["historico", "meio", "tipo", "banco", "contraparte"]].copy()
base["valor_log"] = np.log1p(df["valor"].abs())
num_cols = ["valor_log"]
if USAR_DATA:
    base["dia_semana"] = df["data"].dt.day_of_week
    base["dia_mes"] = df["data"].dt.day
    base["mes"] = df["data"].dt.month
    num_cols += ["dia_semana", "dia_mes", "mes"]

# ---- classes -> números 0..7 ----
codificador_y = LabelEncoder()
y_num = codificador_y.fit_transform(df["categoria"])
classes = list(codificador_y.classes_)
usuarios = df["id_usuario"].to_numpy()

# ---- treino / validação / teste ----
indices = np.arange(len(df))


def escolhe_usuarios(disponiveis, alvo):
    """Grupo de usuários inteiros cuja soma de linhas fica mais perto de `alvo` (fração do total)."""
    linhas = pd.Series(usuarios).value_counts()
    melhor = min((c for k in range(1, len(disponiveis)) for c in combinations(sorted(disponiveis), k)),
                 key=lambda c: abs(linhas[list(c)].sum() / len(usuarios) - alvo))
    return set(melhor)


if DIVISAO == "usuario":
    # usuários inteiros em cada parte, escolhidos para dar ~15% das linhas no teste e ~15% na validação
    # (sortear 20% dos usuários não serve: há usuários com 143 linhas e outros com quase 2.900)
    u_teste = escolhe_usuarios(set(usuarios), 0.15)
    u_val = escolhe_usuarios(set(usuarios) - u_teste, 0.15)
    te = indices[np.isin(usuarios, list(u_teste))]
    va = indices[np.isin(usuarios, list(u_val))]
    tr = indices[~np.isin(usuarios, list(u_teste | u_val))]
else:
    tr_val, te = train_test_split(indices, test_size=0.2, stratify=y_num, random_state=SEMENTE)
    tr, va = train_test_split(tr_val, test_size=0.15, stratify=y_num[tr_val], random_state=SEMENTE)

# ---- pré-processamento ajustado SÓ no treino (sem vazar validação/teste) ----
cats_onehot = ["historico", "meio", "tipo", "banco"] + ([] if USAR_TFIDF else ["contraparte"])
partes = [("cat", OneHotEncoder(min_frequency=5, handle_unknown="infrequent_if_exist"), cats_onehot),
          ("num", MinMaxScaler(), num_cols)]
if USAR_TFIDF:
    partes.insert(0, ("contraparte", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3,
                                                     sublinear_tf=True), "contraparte"))
prep_rede = ColumnTransformer(partes, sparse_threshold=0)  # saída densa: a rede recebe matriz normal

X_tr = prep_rede.fit_transform(base.iloc[tr]).astype(np.float32)
X_va = prep_rede.transform(base.iloc[va]).astype(np.float32)
X_te = prep_rede.transform(base.iloc[te]).astype(np.float32)


# ---- tensores na GPU e DataLoaders ----
def tensores(Xp, idx):
    return torch.tensor(Xp, device=device), torch.tensor(y_num[idx], dtype=torch.long, device=device)


X_train_t, y_train_t = tensores(X_tr, tr)
X_val_t, y_val_t = tensores(X_va, va)
X_test_t, y_test_t = tensores(X_te, te)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=BATCH)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=BATCH)

n_entradas = X_train_t.shape[1]   # tamanho da 1ª camada Linear
n_classes = len(classes)          # tamanho da última camada Linear
# pesos para CrossEntropyLoss(weight=pesos_classes): classe rara pesa mais (calculado só no treino)
contagem = np.bincount(y_num[tr], minlength=n_classes)
pesos_classes = torch.tensor(len(tr) / (n_classes * np.maximum(contagem, 1)), dtype=torch.float32, device=device)

print(f"device: {device} | divisão: {DIVISAO} | TF-IDF: {USAR_TFIDF} | data: {USAR_DATA}")
print(f"n_entradas = {n_entradas} | n_classes = {n_classes}")
print(f"treino {len(tr)} | validação {len(va)} | teste {len(te)} linhas")
if DIVISAO == "usuario":
    print("usuários -> treino:", sorted(set(usuarios[tr])), "| validação:", sorted(set(usuarios[va])),
          "| teste:", sorted(set(usuarios[te])))
print("classes (índice -> nome):", dict(enumerate(classes)))

In [ ]:
# arquitetura: entrada -> 4 -> ReLU -> 8 -> ReLU -> classes
# (n_entradas e n_classes vêm da célula de preparação dos dados)
model = Sequential(
    Linear(n_entradas, 4),
    ReLU(),
    Linear(4, 8),
    ReLU(),
    Linear(8, n_classes),  # sem ativação no fim: a CrossEntropyLoss já aplica o softmax
)
model.to(device)
print(model)

In [ ]:
optimizer = Adam(model.parameters(), lr=0.001)
criterion = CrossEntropyLoss().to(device)  # para dar mais peso às classes raras: CrossEntropyLoss(weight=pesos_classes)
epochs = 300
batch = 8

In [ ]:
# os tensores já foram criados (e estão na GPU) na célula de preparação dos dados
X_train_tensor = X_train_t
y_train_tensor = y_train_t
X_test_tensor = X_test_t
y_test = y_test_t.cpu().numpy()

total_batchs = (len(X_train_tensor) + batch - 1) // batch  # arredonda para cima: o último lote pode ser menor
print(f"treino: {len(X_train_tensor)} linhas | {total_batchs} lotes de {batch} por época")

In [ ]:
def train_model():

    model.train()

    for i_epochs in range(epochs):

        # embaralha a ordem a cada época (os dados vêm agrupados por usuário)
        ordem = torch.randperm(len(X_train_tensor), device=device)
        perda_total = 0.0

        for i_batchs in range(total_batchs):

            inicio = i_batchs * batch
            fim = (i_batchs + 1) * batch
            idx = ordem[inicio:fim]

            batch_x = X_train_tensor[idx]
            batch_y = y_train_tensor[idx]

            optimizer.zero_grad()

            output = model(batch_x)
            loss = criterion(output, batch_y)

            loss.backward()
            optimizer.step()
            perda_total += loss.item() * len(idx)

        if (i_epochs + 1) % 50 == 0 or i_epochs == 0:
            print(f"época {i_epochs + 1:3d}/{epochs} | perda média no treino: {perda_total / len(X_train_tensor):.4f}")

In [ ]:
def eval_model():

    model.eval()

    with torch.no_grad():

        output = model(X_test_tensor)

        y_pred = torch.argmax(output, dim=1)

    y_pred = y_pred.cpu().numpy()

    print(f"Acuraçy Score: {(accuracy_score(y_test, y_pred)*100):.2f}%")
    print(f"F1 Score:  {(f1_score(y_test, y_pred,average="macro")*100):.2f}%")
    print(f"Confusion Matrix (linhas = real, colunas = previsto; ordem = {classes}):")
    print(confusion_matrix(y_test, y_pred))

In [ ]:
train_model()
eval_model()